# Session 4 · Spark on Kaggle: fundamentals → Spark ML

**Runs on Kaggle** – no Java setup on your laptop needed.

| Part | What you do | Data |
|---|---|---|
| **0 · Setup** | Java + PySpark, download data, start Spark, convert a 2 GB CSV – **run it before the break** | – |
| **1 · Spark fundamentals** | DataFrames, transformations vs. actions, plans, shuffles, joins | NYC taxi January 2024, Parquet (~3 M trips) |
| **2 · Spark ML** | data quality, features, Pipeline, CrossValidator, saving models | NYC taxi January 2015, CSV (~12.7 M trips) |

**Setup on Kaggle (once, 3 min):**
1. **Create → New Notebook → File → Import Notebook** → upload this `.ipynb`.
2. Right panel → **Input → + Add Input** → search **"NYC Yellow Taxi Trip Data" by elemento** → add it.
3. Right panel → **Settings → Internet: On** (needs a phone-verified account) · **Accelerator: None**.
4. Run all cells of **Part 0**, then take your break.

---
# Part 0 · Setup (run before the break)

## 0.1 Java 17 + PySpark 4.0
PySpark is a Python front end for a **Java** engine (the JVM). Spark 4 needs Java 17 or newer.
This cell checks both and installs what is missing (~1–2 min the first time).

In [ ]:
import os, re, sys, glob, time, shutil, subprocess
from importlib.metadata import version, PackageNotFoundError

ON_KAGGLE = os.path.exists("/kaggle/input")      # True on Kaggle, False on a laptop

def java_major():
    # Return the major Java version (8, 11, 17, ...) or None if Java is missing
    if shutil.which("java") is None:
        return None
    out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    m = re.search(r'version "(\d+)(?:\.(\d+))?', out)
    if not m:
        return None
    major = int(m.group(1))
    return int(m.group(2)) if major == 1 else major      # "1.8" means Java 8

print("Java found:", java_major())
if ON_KAGGLE and (java_major() or 0) < 17:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
    print("Java now:  ", java_major())

try:
    pyspark_version = version("pyspark")
except PackageNotFoundError:
    pyspark_version = None
if not (pyspark_version or "").startswith("4.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark>=4.0,<4.1"], check=True)
    print("installed pyspark", version("pyspark"),
          "– if you had imported an older pyspark before, restart the kernel once")
else:
    print("pyspark", pyspark_version)

## 0.2 Data
- **Part 1:** January 2024 as Parquet + the taxi zone lookup – downloaded from the NYC TLC website (needs Internet: On).
- **Part 2:** January 2015 as one 2 GB CSV – the Kaggle dataset you added as input.

In [ ]:
import urllib.request

def download(url, path):
    # Download url to path once; skip if the file is already there
    if os.path.exists(path):
        print("✓", os.path.basename(path), "already there"); return
    print("↓", url)
    tmp = path + ".part"
    req = urllib.request.Request(url, headers={"User-Agent": "hok-big-data-course"})
    with urllib.request.urlopen(req, timeout=120) as r, open(tmp, "wb") as fh:
        shutil.copyfileobj(r, fh, length=1 << 20)
    os.replace(tmp, path)                 # only complete files get the final name

if ON_KAGGLE:
    WORK = "/kaggle/working"
    DATA = f"{WORK}/data"
    hits = glob.glob("/kaggle/input/**/yellow_tripdata_2015-01.csv", recursive=True)
    CSV = hits[0] if hits else None
else:   # on a laptop with Java 17+: data from get_data.py, CSV path via environment variable
    WORK = "../data/spark_work"
    DATA = "../data"
    CSV = os.environ.get("TAXI_CSV", "../data/kaggle/nyc-yellow-taxi-trip-data/yellow_tripdata_2015-01.csv")
os.makedirs(DATA, exist_ok=True); os.makedirs(WORK, exist_ok=True)

TLC = "https://d37ci6vzurychx.cloudfront.net"
PARQUET_2024 = f"{DATA}/yellow_tripdata_2024-01.parquet"
ZONES = f"{DATA}/taxi_zone_lookup.csv"
try:
    download(f"{TLC}/trip-data/yellow_tripdata_2024-01.parquet", PARQUET_2024)
    download(f"{TLC}/misc/taxi_zone_lookup.csv", ZONES)
except Exception as e:
    raise RuntimeError(f"Download failed ({e}). On Kaggle: Settings → Internet: On.") from None

assert CSV and os.path.exists(CSV), "Dataset missing: Input → + Add Input → 'NYC Yellow Taxi Trip Data' (elemento)"
print(CSV, f"{os.path.getsize(CSV)/1e9:.2f} GB")

## 0.3 The SparkSession – your entry point
`local[*]` means: driver and executors live in **one** JVM on this machine and use all its cores.
On a cluster only `master` changes – the rest of the code stays the same.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("session-04-spark-on-kaggle")
    .master("local[*]")                                  # all cores of this machine
    .config("spark.driver.memory", "12g" if ON_KAGGLE else "2g")
    .config("spark.sql.shuffle.partitions", "16")      # default 200 is meant for clusters
    .config("spark.sql.session.timeZone", "UTC")       # taxi timestamps have no zone: take them as written
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism,
      "| ANSI mode:", spark.conf.get("spark.sql.ansi.enabled"))

def timed(label, fn):
    # Stopwatch: run fn(), print how long it took, return its result.
    # Call it with a lambda: timed("count", lambda: df.count())
    t0 = time.perf_counter()
    out = fn()
    print(f"{label:<40} {time.perf_counter() - t0:7.2f} s")
    return out

## 0.4 Convert the 2 GB CSV to Parquet (runs during the break)
We need the 2015 data as Parquet in Part 2. Converting takes a few minutes, so we start it now.
A CSV has no types – we write the **schema** ourselves instead of letting Spark guess (`inferSchema`).

In [ ]:
SCHEMA = '''
    VendorID INT, tpep_pickup_datetime TIMESTAMP, tpep_dropoff_datetime TIMESTAMP,
    passenger_count INT, trip_distance DOUBLE,
    pickup_longitude DOUBLE, pickup_latitude DOUBLE, RateCodeID INT, store_and_fwd_flag STRING,
    dropoff_longitude DOUBLE, dropoff_latitude DOUBLE, payment_type INT,
    fare_amount DOUBLE, extra DOUBLE, mta_tax DOUBLE, tip_amount DOUBLE, tolls_amount DOUBLE,
    improvement_surcharge DOUBLE, total_amount DOUBLE
'''
PARQUET_2015 = f"{WORK}/trips_2015_01.parquet"

if os.path.exists(f"{PARQUET_2015}/_SUCCESS"):
    print("✓ already converted")
else:
    timed("CSV → Parquet (action)",
          lambda: spark.read.csv(CSV, header=True, schema=SCHEMA)
                       .write.mode("overwrite").parquet(PARQUET_2015))
    print("✓ converted")

☕ **Break.** When you are back, everybody should see `✓ converted`.

---
# Part 1 · Spark fundamentals (January 2024, ~3 M trips)

## 1.1 Reading Parquet into a DataFrame

> **Predict before you run:** does `spark.read.parquet(...)` read all the data into memory?

In [ ]:
df = spark.read.parquet(PARQUET_2024)       # transformation: nothing is read yet
df.printSchema()                            # metadata from the Parquet footer

In [ ]:
row_count = timed("count()  (action)", lambda: df.count())
print(f"{row_count:,} trips in January 2024 | partitions: {df.rdd.getNumPartitions()}")
df.show(5, truncate=False)
df.select("trip_distance", "fare_amount", "passenger_count").describe().show()

## 1.2 Transformations vs. actions

> **Predict:** which of the lines A–D starts a Spark job?
> ```python
> t1 = df.filter(F.col("trip_distance") > 1.0)          # A
> t2 = t1.select("PULocationID", "trip_distance")        # B
> t3 = t2.groupBy("PULocationID").count()                # C
> t3.show(10)                                            # D
> ```

In [ ]:
t1 = df.filter(F.col("trip_distance") > 1.0)          # A – transformation
print("A: no job yet")
t2 = t1.select("PULocationID", "trip_distance")        # B – transformation
print("B: still no job")
t3 = t2.groupBy("PULocationID").count()                # C – transformation (wide: groupBy)
print("C: still no job")
t3.show(10)                                            # D – ACTION: the job runs now

## 1.3 Core DataFrame operations
`select` · `filter` · `withColumn` · `groupBy` + `agg` · `orderBy`

In [ ]:
# select: choose and rename columns
df.select("tpep_pickup_datetime", F.col("trip_distance").alias("distance_miles"),
          "fare_amount", "PULocationID", "DOLocationID").show(5)

In [ ]:
# filter: keep only plausible trips
clean = (df
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("passenger_count") >= 1))

print(f"original rows : {df.count():>10,}")
print(f"after cleaning: {clean.count():>10,}")

In [ ]:
# withColumn: add a column
enriched = clean.withColumn("fare_per_mile", F.round(F.col("fare_amount") / F.col("trip_distance"), 2))
enriched.select("trip_distance", "fare_amount", "fare_per_mile").show(5)

# groupBy + agg: several aggregates in one pass
by_location = (clean
    .groupBy("PULocationID")
    .agg(F.count("*").alias("trip_count"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
         F.round(F.avg("trip_distance"), 2).alias("avg_distance")))

# orderBy: sort the result
by_location.orderBy(F.col("trip_count").desc()).show(10)

## 1.4 Reading the execution plan

> **Predict:** `by_location` is filter → groupBy → agg. Will the plan contain an `Exchange` (a shuffle)?

| In the plan | Meaning |
|---|---|
| `FileScan parquet` | reading the file – the plan is read **bottom to top** |
| `PushedFilters` | filters handed to the Parquet reader (skips row groups) |
| `HashAggregate` | aggregation – appears **twice**: partial (per partition) and final (after the shuffle) |
| `Exchange` | a **shuffle**: rows move between partitions |
| `BroadcastHashJoin` | small table copied to every executor – no shuffle of the big table |

In [ ]:
by_location.explain()

In [ ]:
filter_only = clean.filter(F.col("trip_distance") > 10)
filter_only.explain()          # no Exchange: filter is a narrow transformation

## 1.5 Task A – Where do most trips start?
**Your turn (5 min):** starting from `clean`, compute per `PULocationID` the number of trips (`trip_count`) and the
average fare rounded to 2 decimals (`avg_fare`), sorted by `trip_count` descending. Show the top 15 and find the
`Exchange` in `explain()`.

In [ ]:
top_locations = (clean
    .groupBy("PULocationID")
    .agg(F.count("*").alias("trip_count")))
    # TODO 1: add a second aggregate: the average fare_amount, rounded to 2 decimals, named "avg_fare"
    # TODO 2: sort by trip_count, highest first

top_locations.show(15)
top_locations.explain()

## 1.6 Task B – Add zone names with a join
The location IDs mean nothing to a human. The zone lookup (265 rows) has `LocationID`, `Borough`, `Zone`.

**Your turn (5 min):** join `top_locations` with `zones` (`PULocationID` = `LocationID`), keep `PULocationID`, `Zone`,
`Borough`, `trip_count`, `avg_fare`, sort by `trip_count` descending and show 15 rows. Then look at `explain()`:
which join strategy did Spark choose – and why?

In [ ]:
zones = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")      # fine for 265 rows
    .csv(ZONES))
zones.printSchema()

In [ ]:
result_with_names = top_locations
# TODO 1: .join(zones, <condition>, "left")   condition: top_locations.PULocationID == zones.LocationID
# TODO 2: .select(...) the columns PULocationID, Zone, Borough, trip_count, avg_fare
# TODO 3: .orderBy(...) trip_count descending

result_with_names.show(15, truncate=False)
result_with_names.explain()

> **Fast groups:** where does the `orderBy` appear in the plan relative to the join? Does it add an `Exchange`?

## 1.7 `collect()` and `toPandas()` – only on small results
Both actions bring **all** rows to the driver. On an aggregated result that's fine; on the raw trips it can crash the driver.

In [ ]:
# DANGEROUS on big data: df.collect() / df.toPandas() pull ALL rows into the driver.
# SAFE: aggregate and limit first, then convert the small result.
top20 = by_location.orderBy(F.col("trip_count").desc()).limit(20).toPandas()

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(top20["PULocationID"].astype(str), top20["trip_count"])
ax.set_xlabel("PULocationID"); ax.set_ylabel("trips")
ax.set_title("Top 20 pickup locations – January 2024")
plt.tight_layout(); plt.show()

---
# Part 2 · Spark ML: how long will my taxi ride take? (January 2015, ~12.7 M trips)

**The task:** predict the trip duration in minutes *at the moment the passenger gets in*.
The 2015 data is the *old* TLC schema: GPS coordinates instead of zone IDs.

## 2.1 Reading a CSV is lazy too

In [ ]:
raw = timed("read.csv  (transformation – lazy)", lambda: spark.read.csv(CSV, header=True, schema=SCHEMA))
n   = timed("count()   (action – reads the file)", lambda: raw.count())
print(f"{n:,} rows | partitions: {raw.rdd.getNumPartitions()} | cores: {spark.sparkContext.defaultParallelism}")

**Your turn (2 min):** how many partitions can run **at the same time** here? How many “waves” of tasks does that make?

## 2.2 CSV vs. Parquet – the conversion from the break

In [ ]:
def dir_mb(path):
    # total size of all files in a folder (a Parquet "file" is a folder of part files)
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e6

trips15 = spark.read.parquet(PARQUET_2015)
print(f"CSV {os.path.getsize(CSV)/1e6:,.0f} MB  ->  Parquet {dir_mb(PARQUET_2015):,.0f} MB")

q = lambda df: df.agg(F.avg("trip_distance")).collect()     # one query, used on two DataFrames
timed("avg(trip_distance) on CSV", lambda: q(raw))
timed("avg(trip_distance) on Parquet", lambda: q(trips15))

## 2.3 Veracity: how dirty is the data?
We compute the trip duration and count problems **in a single pass** (one Spark job, not six).

In [ ]:
trips15 = trips15.withColumn(
    "duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)

dq = trips15.select(
    F.count("*").alias("rows"),
    F.sum((F.col("duration_min") <= 0).cast("int")).alias("duration<=0"),
    F.sum((F.col("duration_min") > 180).cast("int")).alias("duration>3h"),
    F.sum((F.col("trip_distance") <= 0).cast("int")).alias("distance<=0"),
    F.sum((F.col("trip_distance") > 60).cast("int")).alias("distance>60mi"),
    F.sum(((F.col("pickup_longitude") == 0) | (F.col("pickup_latitude") == 0)).cast("int")).alias("pickup_at_0/0"),
    F.sum((~F.col("passenger_count").between(1, 6)).cast("int")).alias("passengers_not_1-6"),
)
dq.show(vertical=True)

## 2.4 Cleaning – and ANSI mode
Spark 4 runs in **ANSI mode** by default: a division by zero stops the job instead of returning `NULL`.
`try_divide` returns `NULL` instead.

In [ ]:
demo = spark.createDataFrame([(3.0, 0.0), (3.0, 1.5)], "miles double, hours double")
try:
    demo.select(F.col("miles") / F.col("hours")).show()
except Exception as e:
    print("ANSI mode stopped the job:", str(e).split(".")[0])
demo.select(F.try_divide("miles", "hours").alias("mph")).show()

In [ ]:
NYC = dict(lon=(-74.30, -73.60), lat=(40.50, 41.00))
in_nyc = lambda lon, lat: F.col(lon).between(*NYC["lon"]) & F.col(lat).between(*NYC["lat"])

plausible = (trips15
    .withColumn("speed_mph", F.try_divide(F.col("trip_distance"), F.col("duration_min") / 60))
    .filter(F.col("duration_min").between(1, 180))
    .filter(F.col("trip_distance").between(0.1, 60))
    .filter(F.col("passenger_count").between(1, 6))
    .filter(in_nyc("pickup_longitude", "pickup_latitude") & in_nyc("dropoff_longitude", "dropoff_latitude"))
    .filter(F.col("speed_mph") < 80))

## 2.5 Feature engineering with built-in functions
Only use information that is known **when the passenger gets in**.

**Your turn (5 min):** complete `haversine_km` – the straight-line distance between pickup and dropoff – using only
`pyspark.sql.functions` (`F.radians`, `F.sin`, `F.cos`, `F.asin`, `F.sqrt`).

$$d = 2R \cdot \arcsin\sqrt{\sin^2\left(\tfrac{\Delta\varphi}{2}\right) + \cos\varphi_1 \cos\varphi_2 \sin^2\left(\tfrac{\Delta\lambda}{2}\right)}, \quad R = 6371\text{ km}$$
($\varphi$ = latitude, $\lambda$ = longitude, both in radians)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    # TODO: build the formula with F.radians, F.sin, F.cos, F.asin, F.sqrt
    #       lat1, lon1, ... are column NAMES (strings) -> use F.col(lat1)
    return F.lit(0.0)    # placeholder so the notebook keeps running – replace it!

In [ ]:
features = (plausible
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("weekday", F.dayofweek("tpep_pickup_datetime"))      # 1 = Sunday ... 7 = Saturday
    .withColumn("haversine_km", haversine_km("pickup_latitude", "pickup_longitude",
                                             "dropoff_latitude", "dropoff_longitude"))
    .select("duration_min", "trip_distance", "haversine_km", "passenger_count", "hour", "weekday",
            "RateCodeID", "VendorID",
            "pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude"))

features.cache()               # mark for caching (transformation) ...
n_clean = timed("cache + count (action)", lambda: features.count())   # ... the action fills the cache
print(f"kept {n_clean:,} of {n:,} rows ({n_clean/n:.1%})")
features.show(3)

> **Why not a Python function (UDF)?** `F.sin`, `F.sqrt`, … run inside the JVM and are optimised by Catalyst. A Python
> UDF ships every row to a Python process and back – often 10× slower.

> **Why is `fare_amount` not a feature?** Write your answer down – it is an exam question.

> **Grey zone: `trip_distance`.** It is the *metered* distance, so strictly it is only known at the end of the ride.
> We use it as a stand-in for the planned route distance a navigation app would give at pickup.
> `haversine_km` is truly known at pickup. (Homework: train without `trip_distance` and compare.)

## 2.6 When are rides slowest?

In [ ]:
by_hour = (features.groupBy("hour")
           .agg(F.count("*").alias("trips"), F.avg("duration_min").alias("avg_min"))
           .orderBy("hour"))
pdf = by_hour.toPandas()                       # 24 rows -> safe to collect
ax = pdf.plot(x="hour", y="avg_min", kind="bar", legend=False, figsize=(9, 3),
              title="Average trip duration by pickup hour")
ax.set_ylabel("minutes");

## 2.7 Spark ML in four words
| Concept | What it does | scikit-learn analogue |
|---|---|---|
| **Transformer** | `.transform(df)` → new DataFrame with extra column(s) | `transform()` |
| **Estimator** | `.fit(df)` → returns a fitted **Model** (= a Transformer) | `fit()` |
| **Pipeline** | chain of stages; itself an Estimator | `sklearn.pipeline.Pipeline` |
| **Evaluator** | computes a metric (RMSE, MAE, R²) from a prediction column | `sklearn.metrics` |

Spark ML wants **all features in one vector column** (`VectorAssembler`).
First, split and set a **baseline**: always predict the average duration of the training set.

In [ ]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

LABEL = "duration_min"
train, test = features.randomSplit([0.8, 0.2], seed=42)

rmse = RegressionEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="rmse")
mean_train = train.agg(F.avg(LABEL)).first()[0]
baseline_rmse = rmse.evaluate(test.withColumn("prediction", F.lit(mean_train)))
print(f"baseline: always predict {mean_train:.1f} min -> RMSE {baseline_rmse:.2f} min")

## 2.8 Pipeline 1: linear regression (+ one-hot encoding)
`hour`, `weekday`, `RateCodeID` and `VendorID` are **categories**, not quantities (hour 23 is not “twice” hour 11.5).
A linear model needs them one-hot encoded.

In [ ]:
CAT = ["hour", "weekday", "RateCodeID", "VendorID"]
NUM = ["trip_distance", "haversine_km", "passenger_count"]

indexer   = StringIndexer(inputCols=CAT, outputCols=[c + "_idx" for c in CAT], handleInvalid="keep")
encoder   = OneHotEncoder(inputCols=[c + "_idx" for c in CAT], outputCols=[c + "_ohe" for c in CAT])
assembler = VectorAssembler(inputCols=NUM + [c + "_ohe" for c in CAT], outputCol="features")
lr        = LinearRegression(featuresCol="features", labelCol=LABEL)

pipe_lr = Pipeline(stages=[indexer, encoder, assembler, lr])

## 2.9 Hyper-parameter tuning with CrossValidator
2 × 2 parameter combinations × 3 folds = **12 model fits** (+1 refit on all training data).
To keep this within a few minutes we tune on a **sample** of the training set.

In [ ]:
SAMPLE_FRACTION = 0.10 if ON_KAGGLE else 0.5
train_s = train.sample(fraction=SAMPLE_FRACTION, seed=42).cache()
print(f"tuning on {train_s.count():,} rows")

grid = (ParamGridBuilder()
        .addGrid(lr.regParam, [0.01, 0.1])           # strength of regularisation
        .addGrid(lr.elasticNetParam, [0.0, 1.0])     # 0 = ridge (L2), 1 = lasso (L1)
        .build())
cv = CrossValidator(estimator=pipe_lr, estimatorParamMaps=grid, evaluator=rmse,
                    numFolds=3, parallelism=2, seed=42)

cv_model = timed("CrossValidator.fit", lambda: cv.fit(train_s))
for params, metric in zip(grid, cv_model.avgMetrics):
    print({p.name: v for p, v in params.items()}, f"-> CV RMSE {metric:.3f}")

**Your turn (5 min):** evaluate the best model (`cv_model.bestModel`) on the **test** set with three metrics:
RMSE, MAE and R². Hint: `rmse.evaluate(pred, {rmse.metricName: "mae"})` overrides the metric for one call.

In [ ]:
def evaluate(model, df, name):
    pred = model.transform(df)
    # TODO: compute rmse, mae and r2 with the evaluator `rmse` and its metricName parameter
    scores = {}
    print(name, scores)
    return scores

results = {"baseline (mean)": {"rmse": baseline_rmse}}
results["linear regression"] = evaluate(cv_model.bestModel, test, "linear regression")

## 2.11 Pipeline 2: gradient-boosted trees (no one-hot needed)
Trees split on thresholds, so they can use `hour` and the raw coordinates directly – and they can learn
“from JFK to Midtown at 5 pm takes long”, which a linear model cannot.

In [ ]:
GBT_FEATURES = NUM + ["hour", "weekday", "RateCodeID",
                      "pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude"]
pipe_gbt = Pipeline(stages=[
    VectorAssembler(inputCols=GBT_FEATURES, outputCol="features"),
    GBTRegressor(featuresCol="features", labelCol=LABEL, maxIter=30, maxDepth=6, seed=42),
])
gbt_model = timed("GBT fit", lambda: pipe_gbt.fit(train_s))
results["gradient-boosted trees"] = evaluate(gbt_model, test, "gradient-boosted trees")

importances = sorted(zip(GBT_FEATURES, gbt_model.stages[-1].featureImportances.toArray()),
                     key=lambda x: -x[1])          # sort pairs by importance, largest first
for name, imp in importances:
    print(f"  {name:<20} {imp:.3f}")

In [ ]:
import pandas as pd
pd.DataFrame(results).T.round(3)

## 2.12 Saving a Spark model – not with pickle
The Python `PipelineModel` is only a thin wrapper around objects that live in the **JVM**. Spark therefore has its own
format: a **folder** with JSON metadata and Parquet files. It can be loaded from Python, Scala or Java.

In [ ]:
MODEL_DIR = f"{WORK}/models/gbt_duration"
gbt_model.write().overwrite().save(MODEL_DIR)

for root, _, files in os.walk(MODEL_DIR):
    depth = root[len(MODEL_DIR):].count(os.sep)
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + "/", [f for f in files if not f.startswith(".")][:3])

reloaded = PipelineModel.load(MODEL_DIR)
same = reloaded.transform(test.limit(5)).select("prediction").collect() == \
       gbt_model.transform(test.limit(5)).select("prediction").collect()
print("reloaded model gives identical predictions:", same)

In [ ]:
features.unpersist(); train_s.unpersist()
spark.stop()

---
## Exam-style questions
**Part 1 – Spark fundamentals**
1. A colleague writes `df2 = df.filter(...)`, `df3 = df2.select(...)`, `df4 = df3.groupBy("PULocationID").count()` and
   says: “Three lines, so Spark ran three operations immediately.” Is she right? When does computation start?
2. A `groupBy("PULocationID").count()` on 8 partitions: why is a shuffle needed, and what do the two `HashAggregate`
   steps in the plan do?
3. `df.filter(F.col("fare_amount") > 50).show(10)` returns 10 rows. Did Spark read only 10 rows? Explain.
4. Joining 3 million trips with the 265-row zone table, the plan shows `BroadcastHashJoin`. Why? What is the benefit
   compared with a sort-merge join, and what could go wrong if you broadcast a very large table?

**Part 2 – Spark ML**

5. `spark.read.csv(...)` returned instantly, `count()` took much longer. Explain with *transformation*, *action* and
   *lazy evaluation*. Why would `inferSchema=True` change the first timing?
6. Why is `fare_amount` (or `total_amount`) **not** allowed as a feature for predicting the trip duration here?
   Name the general problem.
7. Transformer or Estimator: `StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `LinearRegression`?
   What does `Pipeline.fit()` return?
8. The linear model needs one-hot encoding for `hour`, the gradient-boosted trees do not. Why?
9. A `CrossValidator` with 3 × 2 parameter values and `numFolds=5`: how many models are trained? Why did we tune on
   a sample?
10. Your teammate wants to `pickle.dump()` the Spark `PipelineModel`. What do you tell them?
11. Spark 4 runs in ANSI mode. What happens with `F.col("a") / F.col("b")` when `b` is 0, and how do you avoid it?
12. The same task with 2 GB of data on a laptop with 16 GB RAM: Spark, DuckDB or pandas + scikit-learn? Justify.

## Homework
- Finish Part 2 if you did not get there in class.
- Train the gradient-boosted trees **without** `trip_distance` (only `haversine_km`). How much worse does it get?
- Lab C (`03_duckdb_database.ipynb`) on your laptop, including the exam-style questions.